In [1]:
import csv
from pathlib import Path
from statistics import mean, median
from typing import Optional, Dict, List
import fnmatch

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

LIB_OPS = {
    'pandas': [
        'filter_group_pandas_seconds',
        'statistics_pandas_seconds',
        'complex_join_pandas_seconds',
        'timeseries_pandas_seconds',
    ],
    'polars': [
        'filter_group_polars_seconds',
        'statistics_polars_seconds',
        'complex_join_polars_seconds',
        'timeseries_polars_seconds',
    ],
    'duckdb': [
        'filter_group_duckdb_seconds',
        'statistics_duckdb_seconds',
        'complex_join_duckdb_seconds',
        'timeseries_duckdb_seconds',
    ],
}

def fval(x: Optional[str]) -> Optional[float]:
    try:
        return float(x) if x not in (None, '', 'N/A') else None
    except Exception:
        return None

def load_rows(csv_path: Path) -> List[Dict[str, str]]:
    with open(csv_path, newline='', encoding='utf-8') as f:
        rdr = csv.DictReader(f)
        return list(rdr)

def filter_rows_by_hosts(rows: List[Dict[str, str]], hosts: List[str], wildcard: bool = False) -> Dict[str, List[Dict[str, str]]]:
    buckets: Dict[str, List[Dict[str, str]]] = {h: [] for h in hosts}
    for r in rows:
        hn = r.get('hostname')
        if hn is None:
            continue
        for h in hosts:
            if (hn == h) or (wildcard and fnmatch.fnmatch(hn, h)):
                buckets[h].append(r)
    return buckets

def summarize_host(rows: List[Dict[str, str]]) -> Dict:
    mem_total = [fval(r.get('memory_total_gb')) for r in rows]
    mem_total = [v for v in mem_total if v is not None]
    mem_avail = [fval(r.get('memory_available_gb')) for r in rows]
    mem_avail = [v for v in mem_avail if v is not None]

    lib_means: Dict[str, Dict[str, float]] = {}
    for lib, cols in LIB_OPS.items():
        per_row_means: List[float] = []
        for r in rows:
            vals = [fval(r.get(c)) for c in cols]
            vals = [v for v in vals if v is not None]
            if vals:
                per_row_means.append(mean(vals))
        if per_row_means:
            lib_means[lib] = {
                'mean': mean(per_row_means),
                'median': median(per_row_means),
                'n': len(per_row_means),
            }

    overall_per_row: List[float] = []
    for r in rows:
        vals: List[Optional[float]] = []
        for cols in LIB_OPS.values():
            vals.extend([fval(r.get(c)) for c in cols])
        flat = [v for v in vals if v is not None]
        if flat:
            overall_per_row.append(mean(flat))

    return {
        'rows': len(rows),
        'mem_total_mean': mean(mem_total) if mem_total else None,
        'mem_avail_mean': mean(mem_avail) if mem_avail else None,
        'libs': lib_means,
        'overall_mean': mean(overall_per_row) if overall_per_row else None,
        'overall_median': median(overall_per_row) if overall_per_row else None,
    }

def relative_pct(a: Optional[float], b: Optional[float]) -> Optional[float]:
    if a is None or b is None or a == 0:
        return None
    return (a - b) / a * 100.0

def plot_comparison(host_a: str, host_b: str, sum_a: Dict, sum_b: Dict):
    if not HAS_PLOTLY:
        print('Plotly not installed; skipping chart. Run `pip install plotly` to enable.')
        return
    # Build series for per-library mean durations
    libs = list(LIB_OPS.keys())
    a_vals = [sum_a.get('libs', {}).get(lib, {}).get('mean') for lib in libs]
    b_vals = [sum_b.get('libs', {}).get(lib, {}).get('mean') for lib in libs]
    fig = go.Figure()
    fig.add_bar(name=f'{host_a}', x=libs, y=a_vals)
    fig.add_bar(name=f'{host_b}', x=libs, y=b_vals)
    fig.update_layout(title='Per-library mean durations (seconds)', barmode='group', yaxis_title='Seconds')
    fig.show()

In [ ]:
# Configure inputs
CSV_PATH = Path('../data/benchmark_results.csv')
HOST_A = 'ZBookFuryG8'
HOST_B = 'ZBookFuryG9'
WILDCARD = False  # Set to True to allow patterns like 'ZBookFuryG8*'

: 

In [ ]:
# Run comparison
rows = load_rows(CSV_PATH)
buckets = filter_rows_by_hosts(rows, [HOST_A, HOST_B], wildcard=WILDCARD)
sum_a = summarize_host(buckets.get(HOST_A, []))
sum_b = summarize_host(buckets.get(HOST_B, []))

def fmt(x, d=3):
    return 'N/A' if x is None else f'{x:.{d}f}'

print('== Host A ==')
print('Rows:', sum_a['rows'])
print('Mem Total Mean (GB):', fmt(sum_a['mem_total_mean']))
print('Mem Avail Mean (GB):', fmt(sum_a['mem_avail_mean']))
for lib, stats in sum_a['libs'].items():
    print(f

: 